In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** Qiskit Functionsは、IBM Quantum&reg; Premium Plan、Flex Plan、およびOn-Prem（IBM Quantum Platform API経由）Planのユーザーのみが利用できる実験的な機能です。プレビューリリースの状態であり、変更される可能性があります。

## 概要
量子化学において、電子構造問題は電子のシュレーディンガー方程式の解、つまり系の電子の振る舞いを記述する量子波動関数を見つけることに焦点を当てています。これらの波動関数は複素振幅のベクトルであり、各振幅は可能な電子配置の寄与に対応しています。

基底状態は系の最低エネルギー波動関数であり、分子系の研究において特別な重要性を持っています。基底状態を計算する最も正確なアプローチはすべての可能な電子配置を考慮しますが、配置の数が系のサイズとともに指数関数的に増加するため、より大きな系では扱いが困難になります。

Handover Iterative Variational Quantum Eigensolver（HI-VQE）は、分子系の基底状態を正確に推定するための革新的なハイブリッド量子古典手法です。量子ハードウェアと古典コンピューティングを統合し、量子プロセッサを使用して候補となる電子配置を効率的に探索し、古典コンピュータ上で得られた波動関数を計算します。コンパクトでありながら化学的に正確な波動関数を生成することで、HI-VQEは量子化学および材料科学における研究と発見を促進します。

![Qunova HI-VQEアルゴリズムの概要を示す画像](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQEは高精度で基底状態を効率的に推定することで、電子構造問題の計算複雑性を削減します。最も関連性の高い電子配置の中から慎重に選択されたサブセットに焦点を当て、精度と効率の両方を最適化します。

古典コンピュータと量子コンピュータの長所を組み合わせることで、HI-VQEは現在の推定波動関数を反復的に改良・改善します。その独自のサブスペース構築技術により、配置選択をより効率的にし、量子化学シミュレーションにおける計算制御と精度の向上をユーザーにもたらします。

アルゴリズムをより詳しく学びたい場合は、[関連する研究論文をお読みください](https://arxiv.org/abs/2503.06292)。
## 説明
分子系における電子配置の数は、系のサイズとともに指数関数的に増加します。ただし、基底状態などの特定の電子状態では、ごく一部の配置のみが状態のエネルギーに大きく寄与することが一般的です。選択配置相互作用（SCI）法はこのスパース性を利用して、最も関連性の高い配置を特定・集中することで計算コストを削減します。この配置のサブセットをサブスペースと呼びます。

HI-VQEは、分子系を表現するための量子コンピュータの固有の効率性を活用してサブスペース探索を支援します。電子構造問題を高精度で解くために、古典的および量子的サブルーチンを統合しています。既存の量子SCI手法とは異なり、HI-VQEは変分トレーニング、反復的なサブスペース構築、および対角化前の配置スクリーニングを組み合わせることで、量子測定、反復回数、および古典的対角化コストを削減することにより効率を向上させます。したがって、HI-VQEはより多くの量子ビットを必要とする大規模な分子系に適用でき、同程度の精度で問題を解くコストを削減します。

![Qunova HI-VQEアルゴリズムの各ステップの詳細説明を示す画像](../docs/images/guides/qunova-chemistry/description.avif)

系の基底状態を計算するために、HI-VQEはまず古典的な化学パッケージPySCFを使用して、分子ジオメトリやその他の分子情報など、ユーザーが提供した入力から分子表現を生成します。次に、ハイブリッド量子古典最適化ループに入り、含まれる配置の数を最小化しながら基底状態を最適に表現するサブスペースを反復的に改良します。ループは、サブスペースサイズやエネルギーの安定性などの収束基準が満たされるまで継続し、その後、計算された基底状態波動関数とエネルギーが出力されます。これらの結果は、正確なポテンシャルエネルギー面の構築や系のさらなる分析に使用できます。

最適化ループは、量子回路のパラメータを調整して高品質なサブスペースを生成することに焦点を当てています。HI-VQEは3つの量子回路オプションを提供しています：[`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving)、[efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2)、および[LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html)です。最適化はその一般的な適合性から、Hartree-Fock参照状態に近い状態で初期化されます。次に回路が量子デバイス上で実行され、得られた量子状態から配置がサンプリングされてバイナリ文字列として返されます。量子デバイスのノイズにより、サンプリングされた配置の一部は電子数やスピンを保存せず、物理的に無効な場合があります。HI-VQEはこれを[qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview)パッケージの配置回復プロセスを使用して対処しており、ユーザーは無効な配置を修正するか破棄するかを選択できます。

有効な配置は、寄与が最小限と予測されるものを除去するオプションのスクリーニングステップを経ます。これによりサブスペースの次元が削減され、対角化ステップのコストが低下します。スクリーニングが有効な場合、有効な配置から予備のサブスペースハミルトニアンが構築され、非常に緩やかな終了基準で対角化が実行されます。各配置の得られた振幅の精度は低いですが、この反復においてサブスペースから除外する配置を予測するのに効果的であり、高速に計算できます。

選択された配置はサブスペースに追加され、系のハミルトニアンがこのサブスペースに射影されます。サブスペースは反復的に更新され、最も関連性の高い配置が反復をまたいで保持されます。このアプローチは代替手法とは異なり、量子回路が各ステップで完全な基底状態を近似する必要がありません。

次に、サブスペースハミルトニアンが古典的に対角化され、最低固有値とその対応する固有ベクトルが取得されます。これらは基底状態とそのエネルギーの近似を表します。反復を通じてサブスペースの品質が向上するにつれて、計算された基底状態は真の基底状態をより良く近似します。この時点で、計算された基底状態に実質的な寄与をしない配置をサブスペースから除去するための追加のスクリーニングステップを実行できます。このステップにより、次の反復に引き継がれるサブスペースができる限りコンパクトになります。これは対角化から返される振幅に基づいて評価されます。これらの振幅は各配置の計算された基底状態への重要度の寄与を表しています。

収束チェックにより、さらなるトレーニングが結果を改善するかどうかが決定されます。改善する場合は、オプションの古典的拡張ステップが実行され、量子回路パラメータが更新されて計算されたエネルギーをさらに最小化し、プロセスが繰り返されます。古典的拡張ステップはサブスペースの追加配置を生成し、量子デバイスからサンプリングされた配置を補完します。まず対角化結果で最大の振幅を持つ配置を特定し、その後、特定された配置から一重および二重励起の新しい配置を生成します。これらの配置の所望の数がサブスペースに追加されます。

反復が収束したと判断されると、HI-VQEは計算された基底状態（サブスペース内の状態とその基底状態波動関数における振幅の形で）、そのエネルギー、および計算された状態が系のハミルトニアンの固有状態を形成しているかどうかの指標となるエネルギー分散の測定値を返します。

ユーザーは使用する量子回路と各量子回路のショット数を決定でき、サブスペースサイズを制御したり、量子生成された配置を補助するための配置の古典的生成を有効にしたりすることができます。このようにして、ユーザーはHI-VQEの動作を希望するアプリケーションに合わせて調整できます。
## はじめに
まず、[functionへのアクセスをリクエスト](https://forms.office.com/r/zN3hvMTqJ1)してください。
次に、[IBM Quantum&reg; APIキー](http://quantum.cloud.ibm.com/)を使用して認証し、すでにローカル環境に[アカウントを保存](/guides/functions#install-qiskit-functions-catalog-client)していると仮定して、Qiskit Functionを次のように選択してください：

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## 入力
functionが受け付けるすべての入力パラメータについては、以下の表を参照してください。続く[分子オプション](#molecule-options)および[HI-VQEオプション](#hi-vqe-options)のセクションで、それらの引数についてさらに詳しく説明します。
| 名前                   | 型                                                           | 説明                                                                                                                                                                                                                                                                                                 | 必須 | デフォルト                                                                  | 例                                                                                   |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | 文字列または原子と座標ペアを含む構造化リストのいずれかを指定できます。文字列として指定する場合、直交座標形式の分子ジオメトリである必要があります。リストとして指定する場合、原子文字列と座標タプルをそれぞれ含むリストのリストとして指定する必要があります。 | はい      | N/A                                                                      | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` または `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | クエリを実行するバックエンドの名前。                                                                                                                                                                                                                                                                      | はい      | N/A                                                                      | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | 対角化のための最大サブスペース次元。数が完全平方数でない場合は、より少ない状態が使用されます。                                                                                                                                                                                                                                                                    | はい      | N/A                                                                      | `100`                                                                                     |
| max\_expansion\_states | int                                                            | 各反復に含まれる古典的に生成されたCI状態の最大数。                                                                                                                                                                                                                                     | はい      | N/A                                                                      | `10`                                                                                      |
| molecule_options                | dict                                                           | HI-VQEへの入力として使用される分子に関するオプション。詳細については[分子オプション](#molecule-options)セクションを参照してください。                                                                                                                                                                                                                                                | いいえ       | 詳細については[分子オプション](#molecule-options)セクションを参照してください。                                 | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options                | dict                                                           | HI-VQEアルゴリズムの動作を制御するオプション。詳細については[HI-VQEオプション](#hi-vqe-options)セクションを参照してください。                                                                                                                                                                                                                                                | いいえ       | 詳細については[HI-VQEオプション](#hi-vqe-options)セクションを参照してください。                                 | `{"shots": 10_000, "max_iter": 10 }`                               |
### 分子オプション
以下の表は、`molecule_options`辞書に設定できるすべてのキーと値、データ型、およびデフォルト値の詳細を示しています。すべてのキーはオプションです。

| キー                               | 値の型                          | デフォルト値                    | 有効な範囲                                                                                                                                                    | 説明                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | 様々                                                                                                                                                        | 分子系の総正味電荷を指定する整数。デフォルト値は0ですが、任意の整数を指定できます。                                                                                                                                                                                                                                                                                                                                                                                                                              |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | 様々                                                                                                                                                        | 基底タイプを指定する文字列。pyscfに渡されます。（例：`"sto-3g"`、`"3-21g"`、`"6-31g"`、`"cc-pvdz"`）                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"active_orbitals"`               | `List[int]`                         | すべての軌道インデックス。             | 問題に有効な空間軌道インデックス。                                                                                                             | 問題で使用される量子ビット数をnとした場合、[0, n)の区間にある活性軌道インデックスのリスト。指定する場合、frozen_orbitals引数も指定する必要があります。                                                                                                                                                                                                                                                            |
| `"frozen_orbitals"`               | `List[int]`                         | インデックスなし。                      | 活性軌道を除く、問題に有効な空間軌道インデックス。                                                                                  | active_orbitalsと同じ範囲のフリーズ軌道インデックスのリスト。指定する場合、active_orbitalsも指定する必要があります。フリーズされた占有軌道ごとに活性電子の数が2減少するため、占有軌道のみをフリーズする必要があることに注意してください。                                                                                                                                                        |
| `"orbital_coeffs"`                | `List[List[float]]`                 | Hartree-Fock分子軌道。 | 様々。                                                                                                                                                       | 系の電子反発積分の計算に使用される空間軌道の係数。有効な例としては、Hartree-Fock分子軌道、自然軌道、およびAVAS軌道があります。                                                                                                                                                                                                                                                                                                                                   |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True`または`False`                                                                                                                                              | 初期分子計算に点群対称性を呼び出して対称適合軌道基底を構築するために使用します。これらの対称適合軌道は、続くSCF計算の基底関数として使用されます。デフォルト値はFalseです。Trueに設定すると呼び出され、任意の点群が自動的に検出されて使用されます。特定の対称性が割り当てられている場合、例えばsymmetry = "Dooh"のように、分子ジオメトリがこの必要な対称性に従っていない場合はエラーが発生します。 |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | [pyscfドキュメント](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build)を参照してください。                                                      | 検出された対称性のサブグループを生成するために使用できます。symmetryキーワード引数を使用して対称性が指定されている場合、これは効果がありません。                                                                                                                                                                                                                                                                                                                         |
| `"unit"`                          | `str`                               | `"angstrom"`                     | [pyscfドキュメント](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build)を参照してください。                                                      | 原子座標と距離に使用する測定単位を指定します。デフォルトはオングストローム単位を使用します。                                                                                                                                                                                                                                                                                                                                                       |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | [pyscfドキュメント](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build)を参照してください。                                                      | 使用する核モデルを指定します。デフォルトは点核モデルを使用し、その他の値はガウス核モデルを有効にします。関数が指定された場合、ガウス核モデルで核電荷分布値「zeta」を生成するために使用されます。                                                                                                                                                                                                                                                   |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | [pyscfドキュメント](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build)を参照してください。                                                      | 分子内の原子の擬ポテンシャルを指定します。デフォルトはNoneで、擬ポテンシャルが適用されず、すべての電子が計算に明示的に含まれることを示します。                                                                                                                                                                                                                                                                                                                                                |
| `"cart"`                          | `bool`                              | `False`                          | [pyscfドキュメント](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build)を参照してください。                                                      | 計算における角運動量基底関数として直交GTO（ガウス型軌道）を使用するかどうかを指定します。デフォルト値のFalseは球面GTOを使用します。                                                                                                                                                                                                                                                               |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | [pyscfドキュメント](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build)を参照してください。                                                      | 各原子の共線スピン磁気モーメントを設定します。デフォルトはNoneで、各原子のスピンはゼロに初期化されます。                                                                                                                                                                                                                                                                                                                                         |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | H2Oの場合は["H 1s", "O 2p"]など                                                                                                                                                             | AVASスキームに含まれる原子軌道を定義します。[AVASドキュメント](https://arxiv.org/abs/1701.07862)を参照してください。                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_threshold"`                | `float`                             | `0.2`                            | 0.0から2.0の間                                                                                                                                                      | 活性空間に保持される原子軌道（AO）を決定するために使用されるカットオフ値を指定します。                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"`または`"ccsd"`                                                                                                                                            | 自然軌道を準備し、自然軌道占有数（NOON）スキームに基づいて活性軌道を選択するための理論的アプローチを定義します。[NOONsドキュメント](https://doi.org/10.1063/5.0042147)を参照してください。軌道数（および量子ビット数）を制御するために、活性軌道インデックスとフリーズ軌道インデックスの両方を提供する必要があります。                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
### HI-VQEオプション
以下の表は、`hivqe_options`辞書に設定できるすべてのキーと値、データ型、およびデフォルト値の詳細を示しています。すべてのキーはオプションです。

| キー                               | 値の型                          | デフォルト値                    | 有効な範囲                                                                                                                                                    | 説明                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | 1から10,000の間。                                                                                                                                          | 各反復で量子デバイスに使用するショット数。                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| `"max_iter"`                      | `int`                               | `25`                             | 1から50の間。                                                                                                                                              | アンサッツを最適化するために実行する最大反復回数。収束が早期に達成された場合、アルゴリズムはより少ない反復を使用する場合があります。                                                                                                                                                                                                                                                                                                                                                                 |
| `"initial_basis_states"`          | `List[str]`                         | Hartree-Fock状態。          | 問題に必要な量子ビット数と一致するビット数のバイナリ文字列。                                                                 | 以前の結果からの古典的状態でアルゴリズムを再起動するために使用できます。                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"`、`"hea"`、または`"lucj"`                                                                                                                                  | 新しい状態を生成するために最適化する量子アンサッツを指定します。`"epa"`は励起保存アンサッツを選択します。`"hea"`はハードウェア効率の良いアンサッツを選択します。`"lucj"`はlocal unitary cluster Jastrowアンサッツを選択します。                                                                                                                                                                                                                                                       |
| `"convergence_count"`             | `int`                               | `3`                              | 2以上。                                                                                                                                                    | アルゴリズムが収束したとみなされる前に、計算されたエネルギーの大幅な変化なしに経過する必要がある反復回数。                                                                                                                                                                                                                                                                                                                                                         |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | 0より大きく1以下。                                                                                                                                     | 収束チェックの目的で重要と見なされる計算エネルギーの変化の大きさ。                                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True`または`False`                                                                                                                                              | `True`の場合、`convergence_count`回の反復が重大な変化の中断なしに発生して収束条件を満たす必要があります。`False`の場合、最適化プロセス中のいずれかの反復で軽微な変化が発生した後、`convergence_count`回の後にアルゴリズムが停止します。                                                                                                                                                 |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True`または`False`。                                                                                                                                             | `qiskit-addon-sqd`パッケージの配置回復を使用するかどうか。Trueの場合、量子デバイスからサンプリングされた無効な状態が古典的に修正されます。Falseの場合、それらは破棄されます。                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | `"linear"`、`"reverse_linear"`、`"pairwise"`、`"circular"`、`"full"`、または`"sca"`のいずれか。`"lucj"`アンサッツを使用する場合、`"lucj_default"`もオプションです。 | 量子回路内で使用するエンタングルメントスキームを指定します。[LUCJアンサッツのffsim規則](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html)に従った一般的なQiskitの規則を使用します。                                                                                                                                                                                                                                                       |
| `"ansatz_reps"`                   | `int`                               | `2`                              | 0より大きい。                                                                                                                                                | 量子回路における各レイヤーの繰り返し回数。                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | 0以上、1未満。                                                                                                                                   | 対角化後にサブスペースからスクリーニングアウトする状態を決定するための許容値。計算された振幅に基づいてサブスペース状態の包含閾値を指定します。                                                                                                                                                                                                                                                                                                                                  |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | `1e-4`から`1e-1`（両端を含む）。                                                                                                                          | 対角化前にサブスペースからスクリーニングアウトする状態を予測するための許容値。各状態の予測振幅の精度を制御し、値が小さいほどより正確な予測が得られます。                                                                                                                                                                                                                                                                                                                                            |
## 出力
functionは4つのキーと値を持つ辞書を返します。キーと値の内容は以下の表に記載されています：

| キー             | 値の型    | 説明                                                                                                               |
|:----------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`      | `float`       | 分子の近似基底状態エネルギー。                                                                      |
| `"states"`      | `List[str]`   | エネルギーを解くために使用されるサブスペースを形成する選択された行列式。アルファ-ベータ交互形式で表されます。 |
| `"eigenvector"` | `List[float]` | `"states"`で構成されるサブスペースの基底状態に対応する固有ベクトル。                                 |
| `"energy_variance"` | `float` | `"states"`で構成されるサブスペースの基底状態のエネルギー分散で、解の品質の指標を示します。この値は非負であり、値が小さいほどサブスペースの基底状態が系のハミルトニアンの固有状態をより良く近似していることを意味します。 |
| `"energy_history"` | `List[float]` | SPSA最適化プロセスの一部として、ハイブリッド最適化プロセス中の各反復で計算されたエネルギー（計算順）。各反復につき2つのエネルギーが計算されます。 |
## 例
最初の例は、HI-VQEアルゴリズムを使用してNH3分子の基底状態エネルギーを計算する方法を示しています。
#### 分子ジオメトリとオプションの定義
NH3の分子ジオメトリは、各原子を「;」で区切った直交座標で提供されます。

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

追加のオプションを分子系のために以下の辞書形式で定義して提供できます。

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

ジオメトリとオプション入力を使用してfunctionを実行します。

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

問題が発生した場合のサポートリクエストで提供できるように、Function job IDを印刷しておくことをお勧めします。

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


この例では、NH3分子のsto3g基底で8軌道を持つ16量子ビットを使用します。
Qiskit Functionワークロードの[ステータス](/guides/functions#check-job-status)を確認したり、次のように[結果](/guides/functions#retrieve-results)を返したりすることができます：

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


ジョブが完了したら、`result()`インスタンスで結果を取得できます。

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

基底状態エネルギーにアクセスするには、「energy」キーを使用します。「eigenvector」キーは、結果の「states」に格納された電子配置のビット文字列表記に対応するCI係数を提供します。

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

出力：

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936
## パフォーマンス
このセクションでは、Li2Sの24量子ビットケース、N2分子の40量子ビットケース、およびFeP-NOシステムの44量子ビットケースに対するHI-VQEのベンチマーク計算の実証を示します。
#### 24量子ビットのLi2S分子の解離ポテンシャルエネルギー面曲線
PES曲線はFCI参照値とRHFからの初期推定値とともに示され、FCI参照値からのエネルギー誤差も示されています。

![HI-VQEがLi2S系の古典的参照PES曲線に対して化学精度内の解を生成することを示す画像](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

計算は以下のジオメトリとオプションで実施されています。